# Avaliação dos comparativa dos resultados de treinamento

# 1. Imports e definições

In [205]:
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime

# 2. Funções auxiliares

In [206]:
def formatar_data(data_str):
    """
    Recebe uma string de data no formato 'YYYY-MM-DD HH:MM:SS' e retorna a mesma string formatada.
    """
    return datetime.strptime(data_str, "%Y-%m-%d %H:%M:%S").strftime("%Y-%m-%d %H:%M:%S")

def calcular_duracao_minutos(start_time, end_time):
    """
    Calcula a diferença em minutos entre dois datetimes,
    retornando o valor com 3 casas decimais.
    """
    # Garante que ambos são objetos datetime
    start_time = pd.to_datetime(start_time)
    end_time = pd.to_datetime(end_time)
    diff_minutes = (end_time - start_time).total_seconds() / 60
    return round(diff_minutes, 3)

def formatar_parametros(num_str):
    """
    Recebe um número (em string ou int/float), e retorna uma string formatada:
    4058538 -> '4.1M', 7778432 -> '7.8M', 3003079 -> '3.0M', etc.
    """
    try:
        num = float(num_str)
        if num >= 1_000_000:
            return f"{num/1_000_000:.1f}M"
        elif num >= 1_000:
            return f"{num/1_000:.1f}K"
        else:
            return str(int(num))
    except Exception as e:
        print(f'Erro ao formatar {num_str}: {e}')
        return str(num_str)


# 3. Carga dos dados

In [207]:
# Caminho do diretório onde estão os arquivos JSON
diretorio = '../models_results'

df_resultados = pd.DataFrame()

# Itera sobre todos os arquivos e subdiretórios
for root, dirs, files in os.walk(diretorio):
    for file in files:
        if file.endswith('.json'):
            caminho_arquivo = os.path.join(root, file)
            try:
                with open(caminho_arquivo, 'r', encoding='utf-8') as f:
                    dados = json.load(f)
                    dados_df = pd.json_normalize(dados)
                    df_resultados = pd.concat([df_resultados, dados_df])
                    
            except Exception as e:
                print(f'Erro ao ler {caminho_arquivo}: {e}')

# 4. Tratamento dos dados

In [208]:
df_resultados = df_resultados.sort_values(by=['modelo','dataset'])
df_resultados.index = np.arange(0, len(df_resultados))

In [209]:
df_resultados.loc[4,['training_start_time']] = data_formatada = formatar_data("2025-07-19 21:18:16")
df_resultados.loc[4,['training_end_time']] = data_formatada = formatar_data("2025-07-19 21:42:39")

df_resultados.loc[5,['training_start_time']] = data_formatada = formatar_data("2025-07-14 02:40:03")
df_resultados.loc[5,['training_end_time']] = data_formatada = formatar_data("2025-07-14 02:57:39")

df_resultados.loc[4:5,['model_params']] = 7778432.0

df_resultados.loc[:,['model_params']] = df_resultados.loc[:,['model_params']].astype(int).astype(str)

df_resultados['training_duration'] = df_resultados.apply(lambda row: calcular_duracao_minutos(row['training_start_time'], row['training_end_time']), axis=1)

df_resultados.loc[df_resultados['dataset'] == 'dataset_wagner_v4_balanceado',['dataset']] = 'dataset_wagner_balanceado'
df_resultados.loc[df_resultados['dataset'] == 'dataset_wagner_v4_balanceado_1000',['dataset']] = 'dataset_wagner_balanceado_1000'

df_resultados.loc[:, ['model_params']] = df_resultados.loc[:, ['model_params']].apply(lambda x: formatar_parametros(x), axis=1)


/tmp/ipykernel_5514/798362342.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['4058538' '4058538' '6584206' '6584206' '7778432' '7778432' '10794294'
 '10794294' '17686374' '17686374' '28527870' '28527870' '40976278'
 '40976278' '64115614' '64115614' '21817127' '21817127' '3003079'
 '3003079' '23602055' '23602055']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_resultados.loc[:,['model_params']] = df_resultados.loc[:,['model_params']].astype(int).astype(str)
/tmp/ipykernel_5514/1114615219.py:24: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  num = float(num_str)


In [210]:
df_resultados

,dataset,modelo,batch_size,epochs,fine_tuning_epochs,train_acc,valid_acc,train_fine_tuning_acc,valid_finetuning_acc,data_rodada,model_params,training_start_time,training_end_time,training_duration
0,dataset_wagner_balanceado,EfficientNet-B0,32,20,10,0.930952,0.935714,0.964626,0.954762,2025-07-14 02:03:43,4.1M,2025-07-14 01:57:08,2025-07-14 02:03:25,6.283
1,dataset_wagner_balanceado_1000,EfficientNet-B0,32,20,10,0.930408,0.925714,0.971429,0.938571,2025-07-19 20:48:58,4.1M,2025-07-19 20:32:27,2025-07-19 20:48:37,16.167
2,dataset_wagner_balanceado,EfficientNet-B1,32,20,10,0.903061,0.916667,0.963946,0.950000,2025-07-14 02:39:32,6.6M,2025-07-14 02:24:02,2025-07-14 02:38:58,14.933
3,dataset_wagner_balanceado_1000,EfficientNet-B1,32,20,10,0.905306,0.910000,0.968980,0.938571,2025-07-19 21:15:22,6.6M,2025-07-19 20:49:28,2025-07-19 21:14:52,25.400
4,dataset_wagner_balanceado,EfficientNet-B2,32,20,10,0.878571,0.892857,0.958844,0.947619,2025-07-14 02:58:06,7.8M,2025-07-19 21:18:16,2025-07-19 21:42:39,24.383
5,dataset_wagner_balanceado_1000,EfficientNet-B2,32,20,10,0.885102,0.912857,0.962857,0.942857,2025-07-19 21:43:46,7.8M,2025-07-14 02:40:03,2025-07-14 02:57:39,17.600
6,dataset_wagner_balanceado,EfficientNet-B3,32,20,10,0.876531,0.919048,0.951020,0.930952,2025-07-14 03:18:45,10.8M,2025-07-14 02:58:39,2025-07-14 03:18:13,19.567
7,dataset_wagner_balanceado_1000,EfficientNet-B3,32,20,10,0.870204,0.907143,0.962653,0.948571,2025-07-19 22:19:48,10.8M,2025-07-19 21:44:17,2025-07-19 22:19:12,34.917
8,dataset_wagner_balanceado,EfficientNet-B4,32,20,10,0.887075,0.866667,0.964286,0.947619,2025-07-14 03:48:35,17.7M,2025-07-14 03:19:18,2025-07-14 03:47:54,28.600
9,dataset_wagner_balanceado_1000,EfficientNet-B4,32,20,10,0.895306,0.920000,0.973061,0.938571,2025-07-19 23:18:49,17.7M,2025-07-19 22:20:20,2025-07-19 23:18:00,57.667


In [211]:
df_resultados = df_resultados.loc[:,['dataset', 'modelo', 'batch_size', 'epochs', 'fine_tuning_epochs',
       'train_acc', 'valid_acc', 'train_fine_tuning_acc',
       'valid_finetuning_acc', 'model_params', 'training_duration']]

In [212]:
# Dicionário de renomeação de colunas para português
colunas_renomear = {
    'dataset': 'Conjunto de dados',
    'modelo': 'Modelo',
    'batch_size': 'Batch size',
    'epochs': 'Épocas',
    'fine_tuning_epochs': 'Épocas de fine tuning',
    'train_acc': 'Acurácia de treinamento',
    'valid_acc': 'Acurácia de validação',
    'train_fine_tuning_acc': 'Acurácia de treinamento de fine tuning',
    'valid_finetuning_acc': 'Acurácia de validação de fine tuning',
    'model_params': 'Quantidade de Parâmetros (M)',
    'training_duration': 'Duração treinamento (min)'
}

# Renomeando todas as colunas de acordo com o dicionário
df_resultados = df_resultados.rename(columns=colunas_renomear)

df_resultados

,Conjunto de dados,Modelo,Batch size,Épocas,Épocas de fine tuning,Acurácia de treinamento,Acurácia de validação,Acurácia de treinamento de fine tuning,Acurácia de validação de fine tuning,Quantidade de Parâmetros (M),Duração treinamento (min)
0,dataset_wagner_balanceado,EfficientNet-B0,32,20,10,0.930952,0.935714,0.964626,0.954762,4.1M,6.283
1,dataset_wagner_balanceado_1000,EfficientNet-B0,32,20,10,0.930408,0.925714,0.971429,0.938571,4.1M,16.167
2,dataset_wagner_balanceado,EfficientNet-B1,32,20,10,0.903061,0.916667,0.963946,0.950000,6.6M,14.933
3,dataset_wagner_balanceado_1000,EfficientNet-B1,32,20,10,0.905306,0.910000,0.968980,0.938571,6.6M,25.400
4,dataset_wagner_balanceado,EfficientNet-B2,32,20,10,0.878571,0.892857,0.958844,0.947619,7.8M,24.383
5,dataset_wagner_balanceado_1000,EfficientNet-B2,32,20,10,0.885102,0.912857,0.962857,0.942857,7.8M,17.600
6,dataset_wagner_balanceado,EfficientNet-B3,32,20,10,0.876531,0.919048,0.951020,0.930952,10.8M,19.567
7,dataset_wagner_balanceado_1000,EfficientNet-B3,32,20,10,0.870204,0.907143,0.962653,0.948571,10.8M,34.917
8,dataset_wagner_balanceado,EfficientNet-B4,32,20,10,0.887075,0.866667,0.964286,0.947619,17.7M,28.600
9,dataset_wagner_balanceado_1000,EfficientNet-B4,32,20,10,0.895306,0.920000,0.973061,0.938571,17.7M,57.667


In [213]:
df_resultados.to_csv('resultado_comparativo_treinamento.csv', index=False)